In [1]:
!pip -q install sentence-transformers

In [2]:
from google.colab import drive
# Mount Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# 1. INSTALL LIBRARIES AND CONFIGURE PATHS

import json
import time
import numpy as np
import pandas as pd
import torch
import sentence_transformers

from pathlib import Path
from sentence_transformers import CrossEncoder

# Project directory
PROJECT_DIR = Path("/content/drive/MyDrive/NLP Project")

# Previously saved hybrid retrieval results
HYBRID_DIR = PROJECT_DIR / "data/retrieval_results/hybrid"

INPUT_FILE = (
    HYBRID_DIR / "hybrid_multiple_questions.jsonl"
)

# Save reranking outputs separately
OUTPUT_DIR = PROJECT_DIR / "data/retrieval_results/reranked"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"

MAX_LENGTH = 512
BATCH_SIZE = 16
TOP_K = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STRATEGIES = [
    "fixed_size",
    "sentence_based",
    "recursive",
    "section_aware"
]

# Check that hybrid results exist
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Hybrid retrieval results not found: {INPUT_FILE}"
    )

print("Input:", INPUT_FILE)
print("Output:", OUTPUT_DIR)
print("Model:", MODEL_NAME)
print("Device:", DEVICE)


Input: /content/drive/MyDrive/NLP Project/data/retrieval_results/hybrid/hybrid_multiple_questions.jsonl
Output: /content/drive/MyDrive/NLP Project/data/retrieval_results/reranked
Model: cross-encoder/ms-marco-MiniLM-L6-v2
Device: cpu


Load and validate your hybrid retrieval results

In [4]:
# 2. LOAD AND VALIDATE HYBRID RETRIEVAL RESULTS

# Load saved hybrid results
hybrid_df = pd.read_json(
    INPUT_FILE,
    lines=True
)

# Required fields for reranking and citation tracking
required_columns = {
    "question_id",
    "question_type",
    "question",
    "strategy",
    "rank",
    "rrf_score",
    "chunk_id",
    "article_id",
    "title",
    "page_numbers",
    "text"
}

missing = required_columns - set(hybrid_df.columns)

if missing:
    raise ValueError(f"Missing columns: {missing}")

if hybrid_df.empty:
    raise ValueError("No hybrid retrieval results found.")

# Validate questions and passages
assert hybrid_df["question"].map(
    lambda x: isinstance(x, str) and bool(x.strip())
).all()

assert hybrid_df["text"].map(
    lambda x: isinstance(x, str) and bool(x.strip())
).all()

# Check that citation pages are available
assert hybrid_df["page_numbers"].map(
    lambda x: isinstance(x, list) and len(x) > 0
).all()

# Ensure no repeated chunks within one search result
assert not hybrid_df.duplicated(
    ["question_id", "strategy", "chunk_id"]
).any()

# Ensure each search has one consistent question
assert hybrid_df.groupby(
    ["question_id", "strategy"]
)["question"].nunique().eq(1).all()

# Ensure all four chunking strategies are present
assert set(hybrid_df["strategy"]) == set(STRATEGIES)

# Preserve original hybrid ranks
hybrid_df["rank"] = hybrid_df["rank"].astype(int)

assert not hybrid_df.duplicated(
    ["question_id", "strategy", "rank"]
).any()

print("Hybrid results loaded successfully.")
print("Total passages:", len(hybrid_df))
print("Questions:", hybrid_df["question_id"].nunique())
print("Strategies:", hybrid_df["strategy"].nunique())

print("\nCandidates per question and strategy:")

display(
    hybrid_df.groupby(
        ["question_id", "strategy"]
    ).size().unstack()
)


Hybrid results loaded successfully.
Total passages: 480
Questions: 12
Strategies: 4

Candidates per question and strategy:


strategy,fixed_size,recursive,section_aware,sentence_based
question_id,,,,
Q01,10,10,10,10
Q02,10,10,10,10
Q03,10,10,10,10
Q04,10,10,10,10
Q05,10,10,10,10
Q06,10,10,10,10
Q07,10,10,10,10
Q08,10,10,10,10
Q09,10,10,10,10


Load the Cross-Encoder model

In [5]:

# 3. LOAD CROSS-ENCODER MODEL

reranker = CrossEncoder(
    MODEL_NAME,
    max_length=MAX_LENGTH,
    device=DEVICE,
    activation_fn=torch.nn.Identity()
)

print("Cross-Encoder loaded successfully.")
print("Model:", MODEL_NAME)
print("Maximum input length:", MAX_LENGTH)
print("Device:", DEVICE)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder loaded successfully.
Model: cross-encoder/ms-marco-MiniLM-L6-v2
Maximum input length: 512
Device: cpu


Rerank all 480 passages and save results

In [6]:
# 4. RERANK ALL HYBRID RETRIEVAL RESULTS

# 1. Construct question-passage pairs

pairs = list(
    hybrid_df[["question", "text"]].itertuples(
        index=False,
        name=None
    )
)

print("Total question-passage pairs:", len(pairs))

# 2. Generate Cross-Encoder relevance scores

start_time = time.perf_counter()

scores = reranker.predict(
    pairs,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

reranking_time = time.perf_counter() - start_time

# Convert predictions into a 1D array
scores = np.asarray(scores).reshape(-1)

# Validate predictions
assert len(scores) == len(hybrid_df)
assert np.isfinite(scores).all()

# 3. Attach scores to original hybrid results

reranked_df = hybrid_df.copy()

# Preserve the original RRF ranking
reranked_df = reranked_df.rename(
    columns={"rank": "hybrid_rank"}
)

reranked_df["cross_encoder_score"] = scores

# 4. Sort separately for each question and strategy

reranked_df = reranked_df.sort_values(
    by=[
        "question_id",
        "strategy",
        "cross_encoder_score",
        "hybrid_rank"
    ],
    ascending=[True, True, False, True],
    kind="stable"
).reset_index(drop=True)

# Assign the new ranking position
reranked_df["rerank_rank"] = (
    reranked_df.groupby(
        ["question_id", "strategy"]
    ).cumcount() + 1
)
# 5. Select the top five passages

top5_df = reranked_df[
    reranked_df["rerank_rank"] <= TOP_K
].copy()

# 6. Validate result integrity

assert len(reranked_df) == len(hybrid_df)

assert not reranked_df.duplicated(
    ["question_id", "strategy", "chunk_id"]
).any()

# The same chunk IDs must be present before and after reranking
original_keys = set(
    zip(
        hybrid_df["question_id"],
        hybrid_df["strategy"],
        hybrid_df["chunk_id"]
    )
)

reranked_keys = set(
    zip(
        reranked_df["question_id"],
        reranked_df["strategy"],
        reranked_df["chunk_id"]
    )
)

assert original_keys == reranked_keys

# 7. Save full and top-five reranked results

reranked_df.to_json(
    OUTPUT_DIR / "reranked_all_results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

top5_df.to_json(
    OUTPUT_DIR / "reranked_top5_results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

# Save model configuration
config = {
    "reranker_model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "top_k": TOP_K,
    "device": DEVICE,
    "total_pairs": len(pairs),
    "total_questions": hybrid_df["question_id"].nunique(),
    "total_strategies": len(STRATEGIES),
    "reranking_seconds": round(reranking_time, 4),
    "sentence_transformers_version": sentence_transformers.__version__,
    "input_file": str(INPUT_FILE)
}

with open(
    OUTPUT_DIR / "reranking_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(config, file, indent=4)

print("\nCROSS-ENCODER RERANKING COMPLETED")
print("Total passages scored:", len(reranked_df))
print("Top-five evidence records:", len(top5_df))
print("Total inference time:", round(reranking_time, 2), "seconds")
print("Results saved to:", OUTPUT_DIR)


Total question-passage pairs: 480


Batches:   0%|          | 0/30 [00:00<?, ?it/s]


CROSS-ENCODER RERANKING COMPLETED
Total passages scored: 480
Top-five evidence records: 240
Total inference time: 132.13 seconds
Results saved to: /content/drive/MyDrive/NLP Project/data/retrieval_results/reranked


Compare hybrid rankings before and after reranking

In [ ]:
# 5. COMPARE BEFORE AND AFTER RERANKING

comparison_rows = []

for (question_id, strategy), group in reranked_df.groupby(
    ["question_id", "strategy"]
):

    # Original RRF top five
    original_top5 = set(
        group.loc[
            group["hybrid_rank"] <= TOP_K,
            "chunk_id"
        ]
    )

    # New Cross-Encoder top five
    new_top5 = set(
        group.loc[
            group["rerank_rank"] <= TOP_K,
            "chunk_id"
        ]
    )

    # Number of passages shared by both top-five lists
    overlap = len(original_top5 & new_top5)

    # Movement in ranking positions
    average_rank_change = (
        group["hybrid_rank"] - group["rerank_rank"]
    ).abs().mean()

    # Distinct research papers in reranked top five
    top5_group = group[
        group["rerank_rank"] <= TOP_K
    ]

    comparison_rows.append({
        "question_id": question_id,
        "strategy": strategy,
        "top5_overlap": overlap,
        "top5_replacements": TOP_K - overlap,
        "average_rank_change": round(
            average_rank_change, 3
        ),
        "distinct_articles_top5": (
            top5_group["article_id"].nunique()
        )
    })

comparison_df = pd.DataFrame(comparison_rows)

print("QUESTION-LEVEL RERANKING COMPARISON")
display(comparison_df)

# Summarize by chunking strategy
strategy_summary = (
    comparison_df
    .groupby("strategy")
    .agg(
        questions=("question_id", "count"),
        avg_top5_replacements=("top5_replacements", "mean"),
        avg_rank_change=("average_rank_change", "mean"),
        avg_distinct_articles=("distinct_articles_top5", "mean")
    )
    .round(3)
    .reset_index()
)

print("\nSTRATEGY-LEVEL SUMMARY")
display(strategy_summary)

# Save analysis
comparison_df.to_csv(
    OUTPUT_DIR / "reranking_question_comparison.csv",
    index=False
)

strategy_summary.to_csv(
    OUTPUT_DIR / "reranking_strategy_summary.csv",
    index=False
)


QUESTION-LEVEL RERANKING COMPARISON


,question_id,strategy,top5_overlap,top5_replacements,average_rank_change,distinct_articles_top5
0,Q01,fixed_size,3,2,2.8,4
1,Q01,recursive,4,1,2.6,4
2,Q01,section_aware,2,3,3.6,3
3,Q01,sentence_based,4,1,1.6,3
4,Q02,fixed_size,3,2,1.8,5
5,Q02,recursive,3,2,2.8,4
6,Q02,section_aware,3,2,2.6,5
7,Q02,sentence_based,4,1,2.0,4
8,Q03,fixed_size,4,1,1.8,5
9,Q03,recursive,3,2,2.4,5



STRATEGY-LEVEL SUMMARY


,strategy,questions,avg_top5_replacements,avg_rank_change,avg_distinct_articles
0,fixed_size,12,2.000,2.617,4.083
1,recursive,12,2.000,2.483,4.333
2,section_aware,12,2.083,2.833,4.167
3,sentence_based,12,1.583,2.683,4.083


Inspect the reranked passages

In [ ]:
# 6. INSPECT RERANKED PASSAGES FOR ALL QUESTIONS

# Number of top-ranked passages to display
TOP_N = 3

# Get all question IDs in order
question_ids = sorted(reranked_df["question_id"].unique())

for question_id in question_ids:

    # Retrieve the question text
    question_data = reranked_df[
        reranked_df["question_id"] == question_id
    ]

    question = question_data.iloc[0]["question"]

    print("\n" + "=" * 90)
    print(f"QUESTION {question_id}")
    print("Question:", question)
    print("=" * 90)

    # Inspect every chunking strategy
    for strategy in STRATEGIES:

        results = reranked_df[
            (reranked_df["question_id"] == question_id)
            & (reranked_df["strategy"] == strategy)
        ].sort_values("rerank_rank")

        if results.empty:
            print(f"\nNo results for {strategy}")
            continue

        print("\n" + "-" * 75)
        print("STRATEGY:", strategy.upper())
        print("-" * 75)

        # Display top N reranked passages
        for _, row in results.head(TOP_N).iterrows():

            print("\nReranked position:", row["rerank_rank"])
            print("Original hybrid position:", row["hybrid_rank"])

            print(
                "Cross-Encoder score:",
                round(row["cross_encoder_score"], 4)
            )

            print("Article ID:", row["article_id"])
            print("Title:", row["title"])
            print("Pages:", row["page_numbers"])

            if "section" in row.index:
                print("Section:", row["section"])

            print("\nRetrieved passage:")
            print(row["text"][:800])

            print("-" * 75)

print("\nAll questions and strategies inspected successfully.")


QUESTION Q01
Question: How are graph neural networks used for financial fraud detection?

---------------------------------------------------------------------------
STRATEGY: FIXED_SIZE
---------------------------------------------------------------------------

Reranked position: 1
Original hybrid position: 1
Cross-Encoder score: 7.066
Article ID: FD_083
Title: Ensemble of Graph Neural Networks for Enhanced Financial Fraud Detection
Pages: [1]
Section: 

Retrieved passage:
. INTRODUCTION
As financial transactions become more complicated and interconnected, existing fraud detection technologies are reaching their limits in identifying the many patterns and relationships that seems fraudulent [1]. Given this urgent need,
powerful machine learning algorithms are gaining in popularity. These machine learning techniques are designed to
automatically detect fraud tendencies in data by extracting
statistical features from user profiles, behaviours and transaction summaries. The predictions